<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/Copy_of_MC13_LangChain_LlamaIndex_SQL_HR_Agent_Colab_2026_07_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MC13 — Build an HR Agent with LangChain 🤝 LlamaIndex 🤝 SQL
### Decoding Data Science · AI Residency Masterclass 13

**The mission:** your company just hired you to build **DecoBot** — an HR assistant that can answer *any* leave question an employee throws at it.

The catch? The answers live in **two completely different worlds**:

| World | Example question | Where the answer lives |
|---|---|---|
| 📄 **Unstructured** | *"What's our parental leave policy?"* | A PDF nobody reads |
| 🗄️ **Structured** | *"How many days does Fatima in Finance have left?"* | A SQL database with 100 employees |

One brain, two tools. The agent **decides on its own** which tool to use — and sometimes it needs both. That decision loop (**Reason → Act → Observe → repeat**) is what makes it an *agent* and not a chatbot.

**Today's stack (best of both frameworks):**
- 🦙 **LlamaIndex** → builds the RAG tool over your PDF (it's the RAG specialist)
- 🦜 **LangChain v1** → runs the agent loop with `create_agent` (it's the orchestrator)
- 🗄️ **SQLite** → a realistic 100-employee leave database — and yes, **the agent writes its own SQL** 😎

> ⚠️ All employee data below is **fictional sample data** generated for teaching.

## Step 0 — Install the toolkit 🧰
Latest versions, no pins — LangChain v1 + LlamaIndex. **Takes ~60–90s. Grab a coffee ☕**

In [ ]:
!pip install -q -U langchain langchain-openai llama-index llama-index-readers-file

print("\n✅ Installed: LangChain v1 (agent) + LlamaIndex (RAG). No restart needed.")


✅ Installed: LangChain v1 (agent) + LlamaIndex (RAG). No restart needed.


## Step 1 — Your OpenAI key (from Colab Secrets) 🔑
Keep your key **out of the code** — never paste keys in cells you might share:

1. Click the 🔑 **key icon** in the left sidebar.
2. **+ Add new secret** → name it exactly `openai`, paste your key.
3. Toggle **Notebook access ON**.

In [ ]:
from google.colab import userdata
import os, warnings
warnings.filterwarnings("ignore")  # keep the teaching output clean

os.environ["OPENAI_API_KEY"] = userdata.get("openai")
print("✅ Key loaded from Colab Secrets." if os.environ.get("OPENAI_API_KEY") else "❌ No key found — re-check Step 1.")

✅ Key loaded from Colab Secrets.


## Step 2 — Upload YOUR HR policy PDF 📄⬆️
This is where it gets real: instead of hardcoded text, we use an **actual PDF** — just like you would at work.

Run the cell below → a **file picker** appears → choose your HR policy PDF from your computer.
It lands in a `data/` folder, and LlamaIndex will read **everything** inside that folder.

> 🪂 **Safety net:** if you skip the upload (or the picker is cancelled), the cell auto-creates a sample
> policy file so the rest of the notebook still runs. Live demos never break on our watch.

In [ ]:
import os, shutil
from google.colab import files

os.makedirs("data", exist_ok=True)

print("📄 Pick your HR policy PDF (or hit Cancel if you don't want to upload any file)...")
try:
    uploaded = files.upload()          # opens the file picker
except Exception:
    uploaded = {}

for fname in uploaded:
    shutil.move(fname, os.path.join("data", fname))
    print(f"   ✅ Moved '{fname}' → data/{fname}")

print("\n📂 Contents of data/:", os.listdir("data"))

📄 Pick your HR policy PDF (or hit Cancel if you don't want to upload any file)...


Saving DDS_Leave_Policy_v2.pdf to DDS_Leave_Policy_v2.pdf
   ✅ Moved 'DDS_Leave_Policy_v2.pdf' → data/DDS_Leave_Policy_v2.pdf

📂 Contents of data/: ['.ipynb_checkpoints', 'DDS_Leave_Policy_v2.pdf']


In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader

documents = SimpleDirectoryReader(
    input_dir="data",
    required_exts=[".pdf"],
    file_extractor={".pdf": PDFReader()}
).load_data()

print(len(documents))
print(documents[0].text[:1000])


6
DECODING DATA SCIENCE  |  PEOPLE & CULTURE
DDS-HR-POL-001  ·  Leave Policy  ·  v2.0
Internal — Synthetic demo document for AI training purposes. Not legal advice.
Page 1
 Leave Policy
Decoding Data Science (DDS) — People & Culture
Document ID: DDS-HR-POL-001 · Version 2.0 · Effective date: 1 July 2026 · Applies to: All DDS
employees (Dubai, UAE and remote)
This policy defines all leave entitlements at DDS, how leave accrues, how to request it, and how requests are
approved. It replaces Leave Policy v1 (March 2026) in full. Where UAE Federal Decree-Law No. 33 of 2021
(the UAE Labour Law) provides a greater benefit, the law prevails.
Note: this is a synthetic, training-friendly policy created for demos, onboarding simulations, and HR-chatbot prototypes. It is not
legal advice.
1. Leave Entitlements at a Glance
Leave type
Entitlement
Pay
Minimum notice
Evidence required
Annual leave
22 working days per year
Full pay
7 calendar days (see
§3.4)
None
Sick leave
15 working days per year at


## Step 3 — Build the HR database: 100 employees, real SQL 🗄️
A policy PDF answers *"what are the rules?"* — but *"how many days do **I** have left?"* needs **live data**.

We create a **SQLite database** with **100 realistic employees** across 8 departments — names, join dates,
annual & sick balances. A few of them are *dangerously* low on leave days... the agent will find them 👀

*(SQLite ships with Python — this is a real database, zero setup.)*

In [ ]:
import sqlite3, random, datetime

random.seed(42)  # same "random" data for everyone in the room — reproducible

# --- Realistic name pools (a Dubai-flavoured, multicultural workforce) ---
first_names = [
    "Aisha","Fatima","Omar","Khalid","Layla","Zainab","Yusuf","Hassan","Mariam","Salma",
    "Rahul","Priya","Arjun","Sneha","Vikram","Ananya","Rohan","Kavya","Aditya","Meera",
    "John","Sarah","David","Emma","Michael","Olivia","James","Sophia","Daniel","Grace",
    "Maria","Carlos","Ana","Jose","Lucia","Miguel","Elena","Pablo","Isabella","Diego",
    "Chen","Mei","Wei","Ling","Angelo","Jasmine","Marco","Bianca","Kenji","Yuki",
]
last_names = [
    "Khan","Ahmed","Al Mansoori","Hassan","Rahman","Al Farsi","Sheikh","Malik","Hussain","Qureshi",
    "Sharma","Patel","Kumar","Nair","Iyer","Reddy","Gupta","Menon","Kapoor","Desai",
    "Doe","Smith","Johnson","Williams","Brown","Miller","Wilson","Taylor","Anderson","Thomas",
    "Garcia","Rodriguez","Martinez","Lopez","Fernandez","Santos","Cruz","Reyes","Torres","Rivera",
    "Lee","Wong","Tan","Lim","Rossi","Ricci","Tanaka","Sato","Nguyen","Tran",
]
departments = ["Engineering", "Data & AI", "Sales", "Marketing", "Finance", "HR", "Operations", "Customer Success"]

# --- Generate 100 employees (IDs 1001–1100) ---
rows, used_names = [], set()
for emp_id in range(1001, 1101):
    while True:
        name = f"{random.choice(first_names)} {random.choice(last_names)}"
        if name not in used_names:
            used_names.add(name); break
    dept      = random.choice(departments)
    join_date = (datetime.date(2018, 1, 1) + datetime.timedelta(days=random.randint(0, 2700))).isoformat()
    annual_total = 22
    annual_used  = random.choices(range(0, 23), weights=[3]*8 + [5]*8 + [2]*7, k=1)[0]
    sick_total   = 15
    sick_used    = random.randint(0, 9)
    rows.append((emp_id, name, dept, join_date,
                 annual_total, annual_used, annual_total - annual_used,
                 sick_total,   sick_used,   sick_total - sick_used))

# --- Keep the classic MC12 crew at their famous IDs (slides still work!) ---
legacy = {1024: ("Aisha Khan", 10), 1025: ("John Doe", 19), 1026: ("Maria Garcia", 4), 1027: ("David Lee", 2)}
rows = [
    (eid, legacy[eid][0], r[2], r[3], 22, legacy[eid][1], 22 - legacy[eid][1], r[7], r[8], r[9])
    if eid in legacy else r
    for r in rows for eid in [r[0]]
]

# --- Create the database ---
conn = sqlite3.connect("hr.db", check_same_thread=False)
conn.execute("DROP TABLE IF EXISTS employees")
conn.execute("""
CREATE TABLE employees (
    employee_id      INTEGER PRIMARY KEY,
    name             TEXT,
    department       TEXT,
    join_date        TEXT,
    annual_total     INTEGER,
    annual_used      INTEGER,
    annual_remaining INTEGER,
    sick_total       INTEGER,
    sick_used        INTEGER,
    sick_remaining   INTEGER
)""")
conn.executemany("INSERT INTO employees VALUES (?,?,?,?,?,?,?,?,?,?)", rows)
conn.commit()

print(f"✅ hr.db created with {conn.execute('SELECT COUNT(*) FROM employees').fetchone()[0]} employees\n")
print("👀 A quick peek:")
print(f"{'ID':<7}{'Name':<22}{'Department':<18}{'Annual left':<13}{'Sick left'}")
for r in conn.execute("SELECT employee_id, name, department, annual_remaining, annual_total, sick_remaining FROM employees LIMIT 6"):
    print(f"{r[0]:<7}{r[1]:<22}{r[2]:<18}{str(r[3])+'/'+str(r[4]):<13}{r[5]}")

low = conn.execute("SELECT COUNT(*) FROM employees WHERE annual_remaining <= 3").fetchone()[0]
print(f"\n🚨 Teaser: {low} employees have 3 or fewer annual-leave days left. Soon, the agent will hunt them down with SQL it writes itself.")

✅ hr.db created with 100 employees

👀 A quick peek:
ID     Name                  Department        Annual left  Sick left
1001   Chen Malik            Engineering       16/22        13
1002   Bianca Sheikh         Data & AI         13/22        15
1003   Zainab Nair           Marketing         10/22        7
1004   Arjun Ricci           Operations        12/22        11
1005   Aisha Nguyen          Sales             14/22        13
1006   Sneha Nguyen          HR                20/22        14

🚨 Teaser: 10 employees have 3 or fewer annual-leave days left. Soon, the agent will hunt them down with SQL it writes itself.


## Step 4 — Point LlamaIndex at OpenAI 🦙
`Settings` is LlamaIndex's global config: which LLM answers inside the RAG tool, which model turns your PDF into vectors.

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)              # answers inside the RAG tool
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small") # turns your PDF into vectors

print("✅ LlamaIndex → LLM: gpt-4o-mini | Embeddings: text-embedding-3-small")

✅ LlamaIndex → LLM: gpt-4o-mini | Embeddings: text-embedding-3-small


## Step 5 — Turn the PDF into a searchable brain 🧠 (LlamaIndex RAG)
`SimpleDirectoryReader("data")` reads **everything** in your `data/` folder — PDF, txt, docx, it doesn't care.
Then: **ingest → embed → index → query engine**. That's the **R** in RAG, in 3 lines.

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

#documents    = SimpleDirectoryReader("data").load_data()   # reads your uploaded PDF
index        = VectorStoreIndex.from_documents(documents)  # embed + index
query_engine = index.as_query_engine(similarity_top_k=3)   # the retrieval brain

print(f"✅ Indexed {len(documents)} document chunk(s) from data/\n")

# 🔬 Smoke test — RAG only, no agent yet:
print("🔬 Smoke test → 'How many days of paternity leave do I get?'")
print("   →", query_engine.query("How many days of paternity leave do I get?"))

✅ Indexed 6 document chunk(s) from data/

🔬 Smoke test → 'How many days of paternity leave do I get?'
   → You are entitled to 5 working days of paternity leave at full pay, to be taken within 6 months of the child's birth.


In [ ]:
documents

[Document(id_='196aa78c-5c98-4785-abdc-7ca66322669e', embedding=None, metadata={'page_label': '1', 'file_name': 'DDS_Leave_Policy_v2.pdf', 'file_path': '/content/data/DDS_Leave_Policy_v2.pdf', 'file_type': 'application/pdf', 'file_size': 16573, 'creation_date': '2026-07-04', 'last_modified_date': '2026-07-04'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='DECODING DATA SCIENCE  |  PEOPLE & CULTURE\nDDS-HR-POL-001  ·  Leave Policy  ·  v2.0\nInternal — Synthetic demo document for AI training purposes. Not legal advice.\nPage 1\n Leave Policy\nDecoding Data Science (DDS) — People & Culture\nDocument ID: DDS-HR-POL-001 · Version 2.0 · Effec

## Step 6 — Wrap both worlds as LangChain tools 🔧🔧
Here's the handshake between frameworks: LlamaIndex built the RAG engine, and now **LangChain's `@tool`
decorator** turns it into something the agent can call. Same trick for the database.

**The docstring IS the instruction manual** — it's literally how the agent decides which tool to pick.
Write bad docstrings → get a confused agent. Write good ones → magic.


In [ ]:
from langchain.tools import tool

# ---------- TOOL 1: the policy PDF (LlamaIndex RAG inside) ----------
@tool
def hr_policy(question: str) -> str:
    """Answer questions about the company's HR LEAVE POLICY — rules, entitlements,
    procedures, deadlines (e.g. annual leave rules, sick leave, parental leave, carry-over).
    Use this for POLICY questions. Do NOT use it for individual employee balances."""
    return str(query_engine.query(question))

# ---------- TOOL 2: look up a leave balance by employee NAME (simple version) ----------
@tool
def get_leave_balance(name: str) -> str:
    """Look up an employee's leave balance by their name.
    Give it a name like 'Aisha Khan', and it returns their annual and sick leave balance."""

    # Step 1: search the database for this name (case-insensitive, partial match)
    cur = conn.execute(
        "SELECT name, department, annual_remaining, annual_total, sick_remaining, sick_total "
        "FROM employees WHERE name LIKE ?",
        (f"%{name}%",)
    )
    row = cur.fetchone()

    # Step 2: handle "no match found"
    if row is None:
        return f"No employee found with the name '{name}'. Please check the spelling."

    # Step 3: unpack the row into readable variables
    emp_name, dept, annual_left, annual_total, sick_left, sick_total = row

    # Step 4: build a friendly sentence to return to the agent
    return (f"{emp_name} ({dept}) has {annual_left} out of {annual_total} "
            f"annual leave days remaining, and {sick_left} out of {sick_total} "
            f"sick leave days remaining.")

tools = [hr_policy, get_leave_balance]
print("✅ Tools ready:", [t.name for t in tools])

print("\n🔬 Direct test (no agent yet):")
print(get_leave_balance.invoke({"name": "Aisha Khan"}))

✅ Tools ready: ['hr_policy', 'get_leave_balance']

🔬 Direct test (no agent yet):
Aisha Khan (Sales) has 12 out of 22 annual leave days remaining, and 11 out of 15 sick leave days remaining.


## Step 7 — Assemble the agent (LangChain v1: `create_agent`) 🦜🤖
This is the modern LangChain API — **one function**, running on LangGraph under the hood:

```python
agent = create_agent(model=..., tools=..., system_prompt=...)
```

The loop it runs is exactly the pattern from the slides:
**Reason** (which tool? what input?) → **Act** (call it) → **Observe** (read the result) → repeat until it can answer.

The `ask()` helper below streams that loop live so the room can *watch the agent think*.

In [ ]:
import json
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage

agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=tools,
    system_prompt=(
        "You are DecoBot, a friendly HR assistant. "
        "Use hr_policy for questions about leave POLICY and rules. "
        "Use query_leave_database (write a SQL SELECT) for questions about specific employees' "
        "leave balances or statistics across employees. "
        "If a question needs both policy AND data, use both tools before answering. "
        "Be concise and clear."
    ),
)

def _text(msg):
    """Robustly pull text out of a message (v1 content can be str or blocks)."""
    c = msg.content
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        return " ".join(b.get("text", "") for b in c if isinstance(b, dict))
    return str(c)

def ask(question: str):
    """Run the agent and stream its Reason → Act → Observe loop, live."""
    print(f"❓ {question}\n" + "─" * 72)
    final = ""
    for update in agent.stream({"messages": [{"role": "user", "content": question}]},
                               stream_mode="updates"):
        for _node, payload in update.items():
            if not isinstance(payload, dict):
                continue
            for msg in payload.get("messages", []):
                if isinstance(msg, AIMessage):
                    for tc in (msg.tool_calls or []):
                        print(f"🧠 REASON  → the agent decided it needs a tool")
                        print(f"⚡ ACT     → {tc['name']}  {json.dumps(tc['args'])}")
                    t = _text(msg).strip()
                    if t and not msg.tool_calls:
                        final = t
                elif isinstance(msg, ToolMessage):
                    obs = _text(msg).strip()
                    print(f"🔍 OBSERVE [{msg.name}] →\n   " + obs.replace("\n", "\n   ") + "\n")
    print("🟢 FINAL ANSWER:\n", final)
    return final

print("✅ DecoBot is online with tools: hr_policy (RAG) + query_leave_database (SQL)")

✅ DecoBot is online with tools: hr_policy (RAG) + query_leave_database (SQL)


## Step 8 — Round 1: a POLICY question 📄 (RAG tool only)
The agent should reason it needs `hr_policy`, query your PDF, and answer. Watch the loop 👇

In [ ]:
ask("What is our parental leave policy?");

❓ What is our parental leave policy?
────────────────────────────────────────────────────────────────────────
🧠 REASON  → the agent decided it needs a tool
⚡ ACT     → hr_policy  {"question": "What is our parental leave policy?"}
🔍 OBSERVE [hr_policy] →
   Each parent is entitled to 5 working days of paid parental leave within 6 months of the child's birth, in accordance with UAE Labour Law. A notice of 14 calendar days is required, along with a birth certificate.

🟢 FINAL ANSWER:
 Our parental leave policy allows each parent to take 5 working days of paid parental leave within 6 months of the child's birth, in accordance with UAE Labour Law. A notice of 14 calendar days is required, along with a birth certificate.


## Step 9 — Round 2: a DATA question 🗄️ (SQL tool only)
No function was written for "look up Aisha Khan" — the agent has to **write the SQL itself**.
Watch the `ACT` line: that query was composed by the model, live.

In [ ]:
ask("How many annual leave days does Aisha Khan have left?");

❓ How many annual leave days does Aisha Khan have left?
────────────────────────────────────────────────────────────────────────
🧠 REASON  → the agent decided it needs a tool
⚡ ACT     → get_leave_balance  {"name": "Aisha Khan"}
🔍 OBSERVE [get_leave_balance] →
   Aisha Khan (Sales) has 12 out of 22 annual leave days remaining, and 11 out of 15 sick leave days remaining.

🟢 FINAL ANSWER:
 Aisha Khan has 12 out of 22 annual leave days remaining.


In [ ]:
ask("How many annual leave days does Aisha Khan (employee 1024) have left?");

In [ ]:
ask("What's the average annual leave remaining per department, highest first?");

## Step 11 — The boss round: BOTH tools in one question 🌟
One question, two worlds: it needs the **policy rules** (PDF) *and* a **live balance** (SQL).
Count the loops — the agent should go around **twice** before answering.

In [ ]:
ask("Maria Garcia wants to carry over her unused annual leave to next year. "
    "What does the policy say about carry-over, and how many days does she actually have left?");

❓ Maria Garcia wants to carry over her unused annual leave to next year. What does the policy say about carry-over, and how many days does she actually have left?
────────────────────────────────────────────────────────────────────────
🧠 REASON  → the agent decided it needs a tool
⚡ ACT     → hr_policy  {"question": "What is the company's policy on carrying over unused annual leave to the next year?"}
🧠 REASON  → the agent decided it needs a tool
⚡ ACT     → get_leave_balance  {"name": "Maria Garcia"}
🔍 OBSERVE [get_leave_balance] →
   Maria Garcia (Engineering) has 18 out of 22 annual leave days remaining, and 15 out of 15 sick leave days remaining.

🔍 OBSERVE [hr_policy] →
   A maximum of 10 unused annual leave days may be carried over to the next leave year, and this carry-over is automatic without the need for approval. However, any carried-over days must be used by 31 March of the new leave year, or they will expire without compensation.

🟢 FINAL ANSWER:
 According to the policy, 

## Step 12 — Your turn: interrogate DecoBot 🎤
Ideas to try:
- *"Do public holidays count against my annual leave?"* (policy)
- *"How many people in Engineering have fewer than 5 annual days left?"* (SQL)
- *"Who joined most recently, and what's their sick-leave balance?"* (SQL)
- *"I'm employee 1027 and I want to take 10 days off — is that possible per policy and my balance?"* (both!)
- 😈 Trick question: *"What's the gym membership policy?"* — see how it handles what's NOT in the PDF.

Type `quit` to stop.

In [ ]:
while True:
    q = input("\nAsk DecoBot (or 'quit'): ").strip()
    if q.lower() in {"quit", "exit", ""}:
        print("👋 Done.")
        break
    ask(q)


Ask DecoBot (or 'quit'): what is my leave ballance
❓ what is my leave ballance
────────────────────────────────────────────────────────────────────────
🟢 FINAL ANSWER:
 Please provide your name so I can look up your leave balance.

Ask DecoBot (or 'quit'): Aisha Khan
❓ Aisha Khan
────────────────────────────────────────────────────────────────────────
🧠 REASON  → the agent decided it needs a tool
⚡ ACT     → get_leave_balance  {"name": "Aisha Khan"}
🔍 OBSERVE [get_leave_balance] →
   Aisha Khan (Sales) has 12 out of 22 annual leave days remaining, and 11 out of 15 sick leave days remaining.

🟢 FINAL ANSWER:
 Aisha Khan has 12 out of 22 annual leave days and 11 out of 15 sick leave days remaining. If you have any further questions or need more assistance, feel free to ask!

Ask DecoBot (or 'quit'): quit
👋 Done.


---
### What you just built, mapped to the deck 🗺️
| In the stream | In the talk |
|---|---|
| `🧠 REASON` | **Chain of Thought** — the model deciding what to do |
| `⚡ ACT` | **ReAct** — it *acts* by calling a tool (sometimes writing SQL!) |
| `🔍 OBSERVE` | the result it reads back, grounding the next step |
| `hr_policy` tool | **LlamaIndex RAG** over the PDF you uploaded |
| `query_leave_database` tool | **structured data** — a real SQLite DB, agent-written SQL |
| `create_agent(...)` | **LangChain v1** — the agent loop (LangGraph under the hood) |

**Why two frameworks?** Use each for what it's best at: LlamaIndex is the RAG specialist (ingest → embed → retrieve), LangChain is the orchestrator (tools, loop, streaming). In production teams mix them exactly like this.

**To make it real:**
1. Drop your company's *actual* policy PDFs into `data/` — nothing else changes.
2. Point `sqlite3.connect(...)` at your real HR database (Postgres/MySQL via SQLAlchemy) — keep the SELECT-only guard!
3. Add a third tool (`submit_leave_request`) and you've crossed from *answering* to *acting* — that's the agentic era.

— Built for **AI Residency Cohort 11** · Decoding Data Science 🚀

In [ ]:
# ============================================================
# PART 1 — MEMORY: DecoBot remembers who you are 🧠💾
# Same create_agent — just add a checkpointer + thread_id.
# ============================================================
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=tools,                      # same hr_policy + query_leave_database tools
    checkpointer=InMemorySaver(),     # ← THE memory component (one line!)
    system_prompt=(
        "You are DecoBot, a friendly HR assistant for leave questions.\n"
        "IMPORTANT — identity flow:\n"
        "1. If you do not yet know who you are talking to, warmly ask for their "
        "name or employee ID before answering anything personal.\n"
        "2. When they give a name or ID, verify it with query_leave_database "
        "(use WHERE name LIKE '%...%' for names). Then greet them by their full "
        "name, e.g. 'Welcome, Aisha Khan! 👋'.\n"
        "3. From then on, REMEMBER who they are for the whole conversation — "
        "never ask again. When they ask about 'my leave balance', use their "
        "stored employee ID.\n"
        "4. Use hr_policy for policy/rules questions.\n"
        "Be warm, concise and clear."
    ),
)

def chat(message: str, thread_id: str = "demo-1") -> str:
    """One conversational turn — memory lives in the thread_id."""
    result = memory_agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        {"configurable": {"thread_id": thread_id}},
    )
    reply = _text(result["messages"][-1])
    print(f"🧑 You:     {message}")
    print(f"🤖 DecoBot: {reply}\n")
    return reply

# 🔬 Watch the memory work — three turns, ONE thread:
chat("Hi there!")                                   # → it should ask who you are
chat("I'm employee Aisha Khan")                           # → verifies in SQL, welcomes Aisha by name
chat("How many annual leave days do I have left?")  # → no ID given — it REMEMBERS 🎉

🧑 You:     Hi there!
🤖 DecoBot: Hello! 😊 May I have your name or employee ID so I can assist you better?

🧑 You:     I'm employee Aisha Khan
🤖 DecoBot: Welcome, Aisha Khan! 👋 How can I assist you today?

🧑 You:     How many annual leave days do I have left?
🤖 DecoBot: You have 12 out of 22 annual leave days remaining. If you need any more information, feel free to ask!



'You have 12 out of 22 annual leave days remaining. If you need any more information, feel free to ask!'

In [ ]:
!pip install -q -U gradio

In [ ]:

# ============================================================
# ALL TOGETHER IN COLAB: Memory agent + Gradio UI (Gradio 6 ✅)
# Needs only `tools` from Step 6 — everything else is here.
# ============================================================


import gradio as gr
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# ---------- The agent with MEMORY ----------
memory_agent = create_agent(
    model="openai:gpt-4o-mini",
    tools=tools,                      # hr_policy + query_leave_database from Step 6
    checkpointer=InMemorySaver(),     # ← the memory component (one line!)
    system_prompt=(
        "You are DecoBot, a friendly HR assistant for leave questions.\n"
        "IMPORTANT — identity flow:\n"
        "1. If you do not yet know who you are talking to, warmly ask for their "
        "name before answering anything personal.\n"
        "2. When they give a name, verify it with query_leave_database "
        "(use WHERE name LIKE '%...%'). Then greet them by their full name, "
        "e.g. 'Welcome, Aisha Khan!'. If several employees match, ask which one "
        "they are (mention their departments).\n"
        "3. From then on, REMEMBER who they are for the whole conversation — "
        "never ask again. When they ask about 'my leave balance', use their "
        "stored identity.\n"
        "4. Use hr_policy for policy/rules questions. If a question needs both "
        "policy AND their personal data, use both tools before answering.\n"
        "Be warm, concise and clear. Never invent balances — always check the database."
    ),
)

def _text(msg):
    """Pull text out of a message (content can be a string or blocks)."""
    c = msg.content
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        return " ".join(b.get("text", "") for b in c if isinstance(b, dict))
    return str(c)

# ---------- Gradio UI (each browser session = its own memory) ----------
def respond(message, history, request: gr.Request):
    thread_id = request.session_hash if request else "default"
    result = memory_agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        {"configurable": {"thread_id": thread_id}},
    )
    return _text(result["messages"][-1])

demo = gr.ChatInterface(
    fn=respond,
    title="DecoBot — Your HR Leave Assistant",
    description=(
        "Ask about company leave policy, or say hello and DecoBot will get to "
        "know you, look up your leave balance, and remember you for the rest of "
        "the conversation.\n\n"
        "*Powered by a LangChain agent, LlamaIndex RAG over the policy handbook, "
        "and a live employee database. Sample data for demo purposes.*\n\n"
        "**Built by Decoding Data Science · AI Residency**"
    ),
    examples=[
        "Hi! I'd like to check my leave balance",
        "Hello, this is Maria Garcia from Engineering",
        "How many annual leave days do I have left?",
        "Can I carry over my unused days into next year?",
        "What does our parental leave policy say?",
        "I'm planning a two-week holiday in December — do I have enough days, and what's the approval process?",
    ],
)

demo.launch(share=True, debug=True)   # share=True → public link for the room 📱

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dcb13fc08a666972c8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
